In [1]:
import psycopg2
import pandas as pd

# Connect to PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    port="5432",
    database="music_db",
    user="postgres",
    password="postgres321"
)

print("Connected successfully!")

Connected successfully!


In [2]:
# Run a SQL query and load results directly into a pandas dataframe
query = """
    SELECT 
        title,
        artist,
        plays,
        RANK() OVER (ORDER BY plays DESC) as rank
    FROM songs
    ORDER BY rank
"""

df = pd.read_sql(query, conn)
print(df)

                      title           artist  plays  rank
0           Blinding Lights       The Weeknd   2100     1
1              Shape of You       Ed Sheeran   1950     2
2       Rolling in the Deep            Adele   1800     3
3               Billie Jean  Michael Jackson   1500     4
4         Bohemian Rhapsody            Queen   1200     5
5             Enter Sandman        Metallica   1100     6
6   Smells Like Teen Spirit          Nirvana   1100     6
7                Wonderwall            Oasis    990     8
8          Hotel California           Eagles    950     9
9              Superstition    Stevie Wonder    920    10
10            Lose Yourself           Eminem    870    11
11                  Roxanne       The Police    860    12
12              Purple Haze     Jimi Hendrix    780    13


C:\Users\AnnGene\AppData\Local\Temp\ipykernel_3184\1146668647.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [3]:
# Run CTE query from pgAdmin directly in Python
query = """
    WITH popular_songs AS (
        SELECT title, artist, plays, year
        FROM songs
        WHERE plays > 1000
    ),
    ranked_songs AS (
        SELECT *,
            RANK() OVER (ORDER BY plays DESC) as rank
        FROM popular_songs
    )
    SELECT *
    FROM ranked_songs
    WHERE rank <= 3
"""

top3 = pd.read_sql(query, conn)
print("Top 3 most played songs:")
print(top3)

Top 3 most played songs:
                 title      artist  plays  year  rank
0      Blinding Lights  The Weeknd   2100  2019     1
1         Shape of You  Ed Sheeran   1950  2017     2
2  Rolling in the Deep       Adele   1800  2010     3


C:\Users\AnnGene\AppData\Local\Temp\ipykernel_3184\723953776.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  top3 = pd.read_sql(query, conn)


In [4]:
# Always close the connection when done
conn.close()
print("Connection closed.")

Connection closed.


In [8]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import pandas as pd
import os

load_dotenv(r"D:\Code\Learning\.env")

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

with engine.connect() as conn:
    df = pd.read_sql("SELECT * FROM songs ORDER BY plays DESC", conn)
    print(df)

    id                    title           artist  year  plays
0   13          Blinding Lights       The Weeknd  2019   2100
1   14             Shape of You       Ed Sheeran  2017   1950
2   12      Rolling in the Deep            Adele  2010   1800
3    4              Billie Jean  Michael Jackson  1982   1500
4    1        Bohemian Rhapsody            Queen  1975   1200
5   10            Enter Sandman        Metallica  1991   1100
6    3  Smells Like Teen Spirit          Nirvana  1991   1100
7   11               Wonderwall            Oasis  1995    990
8    2         Hotel California           Eagles  1977    950
9    8             Superstition    Stevie Wonder  1972    920
10   5            Lose Yourself           Eminem  2002    870
11   9                  Roxanne       The Police  1978    860
12   7              Purple Haze     Jimi Hendrix  1967    780
